# B2B Customer Churn & Invoice Default Risk Scoring
**Executive Summary, Cost-Based Threshold Optimization & SHAP Explainability**

---

## 1. Executive Business Summary

In B2B recurring-service, distribution, and wholesale businesses, **silent customer churn and late invoice payments directly erode working capital and net margins**. Most commercial and finance teams react only *after* an account stops placing orders or an invoice becomes 60+ days overdue—when recovery rates are lowest.

This notebook demonstrates an end-to-end, leakage-free machine learning pipeline built with **XGBoost** and **Scikit-Learn** that transforms historical billing, order cadence, and payment behavior into **actionable account-level risk probabilities**. Rather than acting as an opaque "black box," the model pairs financial cost-sensitive decision thresholds with **SHAP explainability** so commercial and collections teams know *who* to contact first and *why*.

---

## 2. Key Capabilities & Commercial Impact

* **High-Precision Early Warning:** Ranks B2B accounts by 90-day churn/default probability so account managers and collections teams focus on the highest-exposure segment.
* **Cost-Driven Threshold Optimization:** Standard `0.50` classification cutoffs ignore financial asymmetry. By tuning the decision threshold based on the real cost ratio between *preventive outreach* vs. *losing a B2B client or writing off an unpaid invoice*, the model maximizes net protected margin.
* **Transparent Risk Drivers (SHAP):** Translates complex feature interactions into clear operational signals (e.g., *declining order frequency*, *rising Days Past Due*, or *partial payment ratio*), enabling tailored retention or credit-limit actions.
* **Production-Ready Engineering:** All preprocessing, imputation, and encoding steps are encapsulated inside leakage-free `Pipeline` objects, ensuring deterministic reproducibility for scheduled batch scoring against ERP/CRM exports.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from xgboost import XGBClassifier

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

## 3. Data Ingestion & B2B Portfolio Dataset
To ensure zero external dependencies and full reproducibility, we generate a realistic synthetic B2B portfolio dataset (`N = 2,500` active accounts) reflecting ERP billing, credit utilization, and CRM activity signals, including realistic missing values in auxiliary operational fields.

In [ ]:
def generate_b2b_portfolio_data(n_samples: int = 2500, random_state: int = 42) -> pd.DataFrame:
    """Generate deterministic synthetic B2B billing, order cadence, and credit risk data."""
    rng = np.random.default_rng(random_state)

    account_ids = [f"ACC-{10000 + i}" for i in range(n_samples)]
    sectors = rng.choice(
        ["Wholesale", "Manufacturing", "Logistics", "B2B Services", "Retail Distribution"],
        size=n_samples,
        p=[0.28, 0.22, 0.18, 0.17, 0.15],
    )
    payment_terms = rng.choice(["Net 15", "Net 30", "Net 60"], size=n_samples, p=[0.25, 0.55, 0.20])

    tenure_months = rng.integers(3, 72, size=n_samples)
    monthly_spend_usd = np.round(rng.lognormal(mean=8.2, sigma=0.65, size=n_samples), 2)
    orders_last_90d = rng.poisson(lam=8, size=n_samples) + 1
    order_freq_change_pct = np.round(rng.normal(loc=-2.0, scale=22.0, size=n_samples), 1)
    avg_days_past_due = np.round(np.clip(rng.gamma(shape=1.8, scale=6.5, size=n_samples) - 2, 0, 75), 1)
    partial_payment_ratio = np.round(np.clip(rng.beta(a=1.3, b=6.0, size=n_samples), 0, 1), 3)
    credit_utilization = np.round(np.clip(rng.normal(loc=0.58, scale=0.22, size=n_samples), 0.05, 1.25), 2)
    support_tickets_90d = rng.poisson(lam=1.4, size=n_samples)

    # Latent risk score driven by realistic operational & financial signals
    logit = (
        -2.6
        + 0.055 * avg_days_past_due
        - 0.032 * order_freq_change_pct
        + 2.10 * partial_payment_ratio
        + 1.45 * np.maximum(0, credit_utilization - 0.75)
        + 0.24 * support_tickets_90d
        - 0.015 * tenure_months
        + np.where(payment_terms == "Net 60", 0.35, 0.0)
        + rng.normal(0, 0.35, size=n_samples)
    )
    prob = 1 / (1 + np.exp(-logit))
    high_risk_event = (rng.uniform(0, 1, size=n_samples) < prob).astype(int)

    df = pd.DataFrame({
        "account_id": account_ids,
        "sector": sectors,
        "payment_terms": payment_terms,
        "tenure_months": tenure_months.astype(float),
        "monthly_spend_usd": monthly_spend_usd,
        "orders_last_90d": orders_last_90d.astype(float),
        "order_freq_change_pct": order_freq_change_pct,
        "avg_days_past_due": avg_days_past_due,
        "partial_payment_ratio": partial_payment_ratio,
        "credit_utilization": credit_utilization,
        "support_tickets_90d": support_tickets_90d.astype(float),
        "high_risk_event": high_risk_event,
    })

    # Inject realistic ERP/CRM missingness (~4-6% in non-mandatory fields)
    for col in ["order_freq_change_pct", "credit_utilization", "support_tickets_90d"]:
        mask = rng.uniform(0, 1, size=n_samples) < 0.05
        df.loc[mask, col] = np.nan

    return df

df = generate_b2b_portfolio_data(n_samples=2500, random_state=RANDOM_STATE)
print(f"Dataset shape: {df.shape[0]:,} accounts x {df.shape[1]} features")
print(f"Positive Risk Rate (Churn / >60d Default): {df['high_risk_event'].mean():.1%}")
df.head()

## 4. Exploratory Risk Signal Analysis
Before training, we validate how key behavioral and billing indicators separate healthy accounts (`0`) from at-risk accounts (`1`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

sns.boxplot(data=df, x="high_risk_event", y="avg_days_past_due", ax=axes[0, 0])
axes[0, 0].set_title("Average Days Past Due by Risk Status")
axes[0, 0].set_xlabel("High Risk Event (0 = Healthy, 1 = Churn/Default)")
axes[0, 0].set_ylabel("Avg Days Past Due")

sns.boxplot(data=df, x="high_risk_event", y="order_freq_change_pct", ax=axes[0, 1])
axes[0, 1].set_title("90-Day Order Frequency Change (%) by Risk Status")
axes[0, 1].set_xlabel("High Risk Event (0 = Healthy, 1 = Churn/Default)")
axes[0, 1].set_ylabel("Order Frequency Change (%)")

sns.boxplot(data=df, x="high_risk_event", y="partial_payment_ratio", ax=axes[1, 0])
axes[1, 0].set_title("Partial Payment Ratio by Risk Status")
axes[1, 0].set_xlabel("High Risk Event (0 = Healthy, 1 = Churn/Default)")
axes[1, 0].set_ylabel("Partial Payment Ratio")

risk_by_terms = df.groupby("payment_terms")["high_risk_event"].mean().reset_index()
sns.barplot(data=risk_by_terms, x="payment_terms", y="high_risk_event", ax=axes[1, 1])
axes[1, 1].set_title("Default / Churn Rate by Credit Payment Terms")
axes[1, 1].set_xlabel("Payment Terms")
axes[1, 1].set_ylabel("Observed Risk Rate")

plt.tight_layout()
plt.show()

## 5. Leakage-Free Preprocessing & XGBoost Hyperparameter Tuning
We separate identifiers (`account_id`) and split the portfolio using stratified sampling (`80%` train / `20%` holdout test). All imputation and encoding transformers are fitted strictly inside cross-validation folds to prevent data leakage.

In [ ]:
TARGET = "high_risk_event"
ID_COL = "account_id"

X = df.drop(columns=[TARGET, ID_COL])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

param_distributions = {
    "classifier__n_estimators": [120, 200, 300],
    "classifier__max_depth": [3, 4, 5],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__subsample": [0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.85, 1.0],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__scale_pos_weight": [1.0, round(scale_pos_weight, 2)],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

y_proba_test = best_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_proba_test)
pr_auc = average_precision_score(y_test, y_proba_test)

print(f"Best CV PR-AUC:   {search.best_score_:.4f}")
print(f"Holdout ROC-AUC:  {roc_auc:.4f}")
print(f"Holdout PR-AUC:   {pr_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
RocCurveDisplay.from_predictions(y_test, y_proba_test, ax=axes[0], name="XGBoost Tuned")
axes[0].set_title(f"Holdout ROC Curve (AUC = {roc_auc:.3f})")

PrecisionRecallDisplay.from_predictions(y_test, y_proba_test, ax=axes[1], name="XGBoost Tuned")
axes[1].set_title(f"Holdout Precision-Recall Curve (AP = {pr_auc:.3f})")

plt.tight_layout()
plt.show()

## 6. Financial Cost-Matrix & Threshold Simulation
A default `0.50` probability threshold treats false positives and false negatives as equally costly. In B2B operations:
* **False Negative (`FN`):** Failing to flag a high-risk account results in an estimated **$1,500 USD** average margin/write-off loss.
* **False Positive (`FP`):** Triggering a proactive retention call or credit review costs **$120 USD** in account management time.

Below we simulate decision thresholds from `0.05` to `0.90` to identify the cutoff that minimizes total financial loss.

In [ ]:
COST_FN = 1500.0  # Uncaught churn / invoice default loss (USD)
COST_FP = 120.0   # Proactive account manager / collections outreach cost (USD)

thresholds = np.linspace(0.05, 0.90, 86)
results = []

for th in thresholds:
    preds = (y_proba_test >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    total_cost = fn * COST_FN + fp * COST_FP
    baseline_cost = (y_test == 1).sum() * COST_FN  # Cost with zero intervention
    net_savings = baseline_cost - total_cost
    results.append({
        "threshold": th,
        "total_cost": total_cost,
        "net_savings": net_savings,
        "recall": recall_score(y_test, preds, zero_division=0),
        "precision": precision_score(y_test, preds, zero_division=0),
    })

sim_df = pd.DataFrame(results)
best_row = sim_df.loc[sim_df["net_savings"].idxmax()]
default_row = sim_df.loc[(sim_df["threshold"] - 0.50).abs().idxmin()]

print(f"Default Threshold (0.50) -> Recall: {default_row['recall']:.1%} | Precision: {default_row['precision']:.1%} | Net Savings: ${default_row['net_savings']:,.0f}")
print(f"Optimal Threshold ({best_row['threshold']:.2f}) -> Recall: {best_row['recall']:.1%} | Precision: {best_row['precision']:.1%} | Net Savings: ${best_row['net_savings']:,.0f}")

plt.figure(figsize=(10, 4.5))
plt.plot(sim_df["threshold"], sim_df["net_savings"], linewidth=2.2, label="Net Financial Savings (USD)")
plt.axvline(best_row["threshold"], color="green", linestyle="--", label=f"Optimal Cutoff ({best_row['threshold']:.2f})")
plt.axvline(0.50, color="gray", linestyle=":", label="Standard Cutoff (0.50)")
plt.title("Financial Impact Simulation Across Probability Thresholds")
plt.xlabel("Classification Probability Threshold")
plt.ylabel("Net Protected Margin vs. No Action (USD)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Executive Explainability (SHAP) & Prioritized Action List
Finally, we extract global risk drivers using **SHAP** and produce an operational watchlist of the top 10 highest-risk B2B accounts for immediate commercial or credit follow-up.

In [ ]:
fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_xgb = best_model.named_steps["classifier"]

feature_names = fitted_preprocessor.get_feature_names_out()
clean_feature_names = [name.replace("num__", "").replace("cat__", "") for name in feature_names]
X_test_transformed = pd.DataFrame(fitted_preprocessor.transform(X_test), columns=clean_feature_names)

if SHAP_AVAILABLE:
    explainer = shap.TreeExplainer(fitted_xgb)
    shap_values = explainer.shap_values(X_test_transformed)
    plt.figure(figsize=(10, 5))
    shap.summary_plot(shap_values, X_test_transformed, plot_type="bar", max_display=10, show=False)
    plt.title("Top 10 Global B2B Risk Drivers (Mean |SHAP Value|)")
    plt.tight_layout()
    plt.show()
else:
    importances = pd.Series(fitted_xgb.feature_importances_, index=clean_feature_names).sort_values(ascending=True).tail(10)
    plt.figure(figsize=(10, 5))
    importances.plot(kind="barh")
    plt.title("Top 10 Global B2B Risk Drivers (XGBoost Gain Importance)")
    plt.xlabel("Relative Feature Importance")
    plt.tight_layout()
    plt.show()

# Generate prioritized watchlist for commercial & collections teams
watchlist = df.loc[X_test.index, ["account_id", "sector", "payment_terms", "monthly_spend_usd", "avg_days_past_due", "order_freq_change_pct"]].copy()
watchlist["risk_probability"] = np.round(y_proba_test, 3)
watchlist["risk_tier"] = pd.cut(
    watchlist["risk_probability"],
    bins=[-0.01, best_row["threshold"], 0.65, 1.01],
    labels=["Low Risk", "Elevated - Monitor", "Critical - Immediate Action"],
)

watchlist.sort_values(by="risk_probability", ascending=False).head(10)